# LDPC: classical to quantum

In [ ]:
import numpy as np
import stim

def make_circulant_row(n, weight=3):
    """
    Generate a weight-w circulant row of length n with good girth properties.
    Places ones at positions 0, 1, and a spread-out third position to avoid short cycles.
    """
    row = np.zeros(n, dtype=int)
    row[0] = 1
    row[1] = 1
    # Third 1 at roughly n//3 to maximize girth
    row[n // 3] = 1
    if weight > 3:
        row[2 * n // 3] = 1
    return row

def circulant(row, n):
    """Build a circulant matrix from first row."""
    H = np.zeros((n, n), dtype=int)
    for i in range(n):
        H[i] = np.roll(row, i)
    return H

# Hamming-like circulant LDPC (connects your notebook narrative)
row = np.array([1, 1, 0, 1, 0, 0, 0])
H1 = circulant(row, 7)  # 7x7 circulant
H2 = H1.copy()          # symmetric construction
print(H1)

# Hypergraph product stabilizer matrices
def hypergraph_product(H1, H2):
    r1, n1 = H1.shape
    r2, n2 = H2.shape
    Ir1 = np.eye(r1, dtype=int)
    Ir2 = np.eye(r2, dtype=int)
    In1 = np.eye(n1, dtype=int)
    In2 = np.eye(n2, dtype=int)

    Hx = np.block([np.kron(H1, In2), np.kron(Ir1, H2.T)])
    Hz = np.block([np.kron(In1, H2), np.kron(H1.T, Ir2)])
    return Hx % 2, Hz % 2

Hx, Hz = hypergraph_product(H1, H2)
print(f"Hx shape: {Hx.shape}, Hz shape: {Hz.shape}")
# Verify CSS condition: Hx @ Hz.T == 0 mod 2
print("CSS condition satisfied:", np.all((Hx @ Hz.T) % 2 == 0))

## Generalized $H_x$

In [ ]:
# !pip install ldpc

In [ ]:
from ldpc.mod2 import nullspace, row_span, rank

def find_logical_x(Hx: np.ndarray, Hz: np.ndarray) -> np.ndarray:
    """
    Find a valid logical X operator: ker(Hx) \\ im(Hz.T)
    """
    assert Hx.shape[1] == Hz.shape[1], (
        f"Column mismatch: Hx has {Hx.shape[1]} cols, Hz has {Hz.shape[1]} cols"
    )
    
    kernel = nullspace(Hx).toarray().astype(int)
    rHz = rank(Hz)          

    for v in kernel:
        v = np.array(v).flatten()  # ensure 1D
        if v.shape[0] != Hz.shape[1]:
            # Pad or raise — should not happen if CSS is valid
            raise ValueError(f"v length {v.shape[0]} != Hz cols {Hz.shape[1]}")
        combined = np.vstack([Hz, v[np.newaxis, :]]) % 2
        if rank(combined) > rHz:
            return v

    raise ValueError("No logical X operator found!")


## Circuit funcs

In [ ]:
def css_x_memory(Hx: np.ndarray, p: float, Hz: np.ndarray) -> stim.Circuit:
    """
    Build a stim X-memory circuit from any binary parity-check matrix Hx.
    
    Hx:  (r x n) binary matrix — each row is one Z-stabilizer (detects X errors)
    p:   X error probability on data qubits
    Hz:  (optional) Z parity-check matrix — used to find true logical X operator
    """
    r, n = Hx.shape
    data = list(range(n))
    anc  = list(range(n, n + r))

    c = stim.Circuit()
    c.append("R", data + anc)
    c.append("X_ERROR", data, p)

    for i, row in enumerate(Hx):
        support = [j for j, val in enumerate(row) if val == 1]
        a = anc[i]
        c.append("H", a)
        for q in support:
            c.append("CZ", [a, q])
        c.append("H", a)
        c.append("M", a)
        c.append("DETECTOR", [stim.target_rec(-1)])

    c.append("M", data)

    # Use true logical X operator if Hz is provided, else fall back to Hx[0]
    logical_vec = find_logical_x(Hx, Hz)

    logical_support = [j for j, val in enumerate(logical_vec) if val == 1]
    c.append(
        "OBSERVABLE_INCLUDE",
        [stim.target_rec(-(n - q)) for q in sorted(logical_support)],
        0
    )
    return c

In [ ]:
import sinter
def make_css_x_memory_experiment(n, p, weight=3):
    """Build a sinter.Task for a circulant hypergraph-product code at error rate p."""
    row = make_circulant_row(n, weight)
    H = circulant(row, n)
    Hx, Hz = hypergraph_product(H, H)
    circ = css_x_memory(Hx, p=p, Hz=Hz)  # Hz passed → find_logical_x() is used
    return sinter.Task(
        circuit=circ,
        json_metadata={"n": n, "p": p, "n_physical": circ.num_qubits}
    )


In [ ]:
import matplotlib.pyplot as plt
import os

## Execution

In [ ]:
os.chdir("/home/ruzza/RepoRoot/QPFL_Hackathon_2026_Alice_and_Bob_Quantum_Icing/team-quantum_icing/core")
print("cwd:", os.getcwd())  # verify it exists

# Sweep over code sizes and error rates
sizes = [5, 7, 9, 11, 13]
error_rates = [0.01, 0.03, 0.1]   # 3 points is enough to see the curve

for n in sizes:
    for p in error_rates:
        try:
            t = make_css_x_memory_experiment(n, p)
            tasks.append(t)
        except ValueError as e:
            print(f"Skipping {n},{p}: {e}")

print("Start...")
# Sample using sinter
results = sinter.collect(
    tasks=tasks,
    max_shots=100,   # quick test — raise to 10_000 once it works
    max_errors=10,
    num_workers=4,
    decoders=["pymatching"],
)

print("DONE!")

In [ ]:
# Plot logical error rate vs physical error rate for each code size
fig, ax = plt.subplots(figsize=(8, 5))
for n in sizes:
    pts = [r for r in results if r.json_metadata["n"] == n]
    pts.sort(key=lambda r: r.json_metadata["p"])
    xs = [r.json_metadata["p"] for r in pts]
    ys = [r.errors / r.shots if r.shots > 0 else 0 for r in pts]
    ax.plot(xs, ys, marker="o", label=f"n={n}")

ax.set_xlabel("Physical error rate p")
ax.set_ylabel("Logical error rate")
ax.set_title("Hypergraph-product circulant codes: X-memory experiment")
ax.legend()
ax.set_xscale("log")
ax.set_yscale("log")
plt.tight_layout()
plt.show()
